# Task 2: Original LCS System on raw Dataset

### Import libs

In [ ]:
# Imports

!python -m pip install matplotlib pandas numpy scikit-learn scikit-eLCS
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

### Load dataset

In [ ]:
# Load in dataset

df = pd.read_csv("cleaned_bank_fraud.csv", header=None, na_values="?")

# Metadata about the data frame
print(f"Shape: {df.shape}")
print(f"Head: {df.head()}")
print(f"Data types: {df.dtypes.value_counts()}")

### Importing Original Data Set and Finding Missing Cells

In [ ]:
raw_df = pd.read_csv("original_dataset/bank_fraud.csv")

print(f"Raw shape: {raw_df.shape}")
print(f"Cleaned shape: {df.shape}")

print(raw_df.columns.tolist())
print(raw_df.isna().sum().sum(), "missing cells")

### 2.1 Processing 

In [ ]:
from sklearn.model_selection import train_test_split

TARGET = "Fraud_Label"   

data = raw_df.dropna(subset=[TARGET]).copy()
data = data.drop(columns=["Transaction_ID", "Customer_ID"])  


for col in data.select_dtypes(include="object").columns:
    codes, _ = pd.factorize(data[col])
    data[col] = codes
    data.loc[data[col] == -1, col] = np.nan   

sample = data.groupby(TARGET, group_keys=False).apply(
    lambda g: g.sample(frac=5000/len(data), random_state=42))

X = sample.drop(columns=[TARGET]).values
y = sample[TARGET].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, np.bincount(y_train))

### Running eLCS



In [ ]:
from skeLCS import eLCS
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, confusion_matrix

model = eLCS(learning_iterations=20000, N=1000)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Balanced acc:", balanced_accuracy_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print(confusion_matrix(y_test, pred))

# Task 3: Data Preprocessing and Feature Engineering

In [3]:
# Task 3 - Data Preprocessing and Feature Engineering
# Loading the original Phase I dataset for data quality assessment
import pandas as pd
import numpy as np

task3_df = pd.read_csv("original_dataset/FraudShield_Banking_Data (1).csv")

print("Task 3 - Initial Dataset Check")
print("--------------------------------")
print(f"Rows: {task3_df.shape[0]}")
print(f"Columns: {task3_df.shape[1]}")

print("\nData types:")
print(task3_df.dtypes)

print("\nMissing values by column:")
print(task3_df.isnull().sum())

print("\nTotal duplicate rows:")
print(task3_df.duplicated().sum())

Task 3 - Initial Dataset Check
--------------------------------
Rows: 50000
Columns: 25

Data types:
Transaction_ID                           float64
Customer_ID                              float64
Transaction_Amount (in Million)          float64
Transaction_Time                          object
Transaction_Date                          object
Transaction_Type                          object
Merchant_ID                              float64
Merchant_Category                         object
Transaction_Location                      object
Customer_Home_Location                    object
Distance_From_Home                       float64
Device_ID                                float64
IP_Address                                object
Card_Type                                 object
Account_Balance (in Million)             float64
Daily_Transaction_Count                  float64
Weekly_Transaction_Count                 float64
Avg_Transaction_Amount (in Million)      float64
Max_Transaction_L

In [4]:
# Checking the missing values in the original dataset

missing_values = task3_df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("Columns with Missing Values")
print("---------------------------")
print(missing_values)

print(f"\nTotal missing cells: {missing_values.sum()}")
print(f"Columns affected: {len(missing_values)}")

Columns with Missing Values
---------------------------
Failed_Transaction_Count                 11
Customer_ID                              10
Transaction_Amount (in Million)           9
Transaction_Time                          9
Merchant_Category                         9
Device_ID                                 9
Avg_Transaction_Amount (in Million)       9
Account_Balance (in Million)              9
Daily_Transaction_Count                   9
Merchant_ID                               7
IP_Address                                6
Transaction_Location                      6
Weekly_Transaction_Count                  5
Is_New_Merchant                           5
Is_International_Transaction              4
Max_Transaction_Last_24h (in Million)     4
Fraud_Label                               4
Customer_Home_Location                    4
Transaction_Type                          4
Card_Type                                 3
Transaction_Date                          3
Unusual_Time_Transac

In [5]:
# Task 3 - Checks categorical columns for invalid or inconsistent values

categorical_columns = task3_df.select_dtypes(include="object").columns

print("Categorical Columns")
print("-------------------")
print(categorical_columns.tolist())

print("\nUnique Value Counts")
print("-------------------")

for column in categorical_columns:
    print(f"{column}: {task3_df[column].nunique(dropna=False)} unique values")

Categorical Columns
-------------------
['Transaction_Time', 'Transaction_Date', 'Transaction_Type', 'Merchant_Category', 'Transaction_Location', 'Customer_Home_Location', 'IP_Address', 'Card_Type', 'Is_International_Transaction', 'Is_New_Merchant', 'Unusual_Time_Transaction', 'Fraud_Label']

Unique Value Counts
-------------------
Transaction_Time: 1441 unique values
Transaction_Date: 122 unique values
Transaction_Type: 4 unique values
Merchant_Category: 7 unique values
Transaction_Location: 11 unique values
Customer_Home_Location: 6 unique values
IP_Address: 49995 unique values
Card_Type: 3 unique values
Is_International_Transaction: 3 unique values
Is_New_Merchant: 3 unique values
Unusual_Time_Transaction: 3 unique values
Fraud_Label: 3 unique values


In [8]:
# Inspects actual values in categorical columns with a small number of categories

category_check_columns = [
    "Transaction_Type",
    "Merchant_Category",
    "Transaction_Location",
    "Customer_Home_Location",
    "Card_Type",
    "Is_International_Transaction",
    "Is_New_Merchant",
    "Unusual_Time_Transaction",
    "Fraud_Label"
]

print("Categorical Value Inspection")
print("----------------------------")

for column in category_check_columns:
    print(f"\n{column}:")
    print(task3_df[column].value_counts(dropna=False))

Categorical Value Inspection
----------------------------

Transaction_Type:
Transaction_Type
Online    16713
ATM       16682
POS       16601
NaN           4
Name: count, dtype: int64

Merchant_Category:
Merchant_Category
Restaurant     8483
ATM            8401
Fuel           8358
Clothing       8281
Grocery        8252
Electronics    8216
NaN               9
Name: count, dtype: int64

Transaction_Location:
Transaction_Location
Multan          5083
Kuala Lumpur    5071
Lahore          5050
Singapore       5025
Islamabad       5019
Faisalabad      5017
Bangkok         4986
Karachi         4931
Dubai           4916
London          4896
NaN                6
Name: count, dtype: int64

Customer_Home_Location:
Customer_Home_Location
Lahore        10118
Islamabad     10016
Karachi       10008
Multan         9948
Faisalabad     9906
NaN               4
Name: count, dtype: int64

Card_Type:
Card_Type
Debit     25106
Credit    24891
NaN           3
Name: count, dtype: int64

Is_International_Tra

In [10]:
# Checks for common invalid values and inconsistent text formatting

invalid_markers = ["?", "unknown", "Unknown", "UNKNOWN", "N/A", "NA", "null", "NULL", ""]

print("Invalid / Inconsistent Value Check")
print("----------------------------------")

for column in categorical_columns:
    invalid_count = task3_df[column].isin(invalid_markers).sum()

    if invalid_count > 0:
        print(f"{column}: {invalid_count} suspicious values")

print("\nWhitespace consistency check:")

for column in categorical_columns:
    non_null = task3_df[column].dropna().astype(str)
    whitespace_count = (non_null != non_null.str.strip()).sum()

    if whitespace_count > 0:
        print(f"{column}: {whitespace_count} values contain leading/trailing spaces")

Invalid / Inconsistent Value Check
----------------------------------

Whitespace consistency check:


In [11]:
# Task 3 - Checks numerical columns for invalid negative values

numeric_columns = task3_df.select_dtypes(include=np.number).columns

print("Numerical Data Validation")
print("-------------------------")

print(f"Numerical columns: {len(numeric_columns)}")

for column in numeric_columns:
    negative_count = (task3_df[column] < 0).sum()

    if negative_count > 0:
        print(f"{column}: {negative_count} negative values")

print("\nNumerical ranges:")
print(task3_df[numeric_columns].agg(["min", "max"]).T)

Numerical Data Validation
-------------------------
Numerical columns: 13

Numerical ranges:
                                            min       max
Transaction_ID                         100043.0  999992.0
Customer_ID                             10005.0   99996.0
Transaction_Amount (in Million)             1.0       9.0
Merchant_ID                             10001.0   99996.0
Distance_From_Home                          1.0     599.0
Device_ID                              100053.0  999989.0
Account_Balance (in Million)                3.0      39.0
Daily_Transaction_Count                     1.0       7.0
Weekly_Transaction_Count                    1.0      24.0
Avg_Transaction_Amount (in Million)         1.0       5.0
Max_Transaction_Last_24h (in Million)       1.0       9.0
Failed_Transaction_Count                    0.0       2.0
Previous_Fraud_Count                        0.0       1.0


# Task 4: Improved LCS-Based System

# Task 5: Experiments

# Task 6: Model Comparison